# 5. Model Monitoring

A model is only correct about the world it was trained on. When that world moves, the model
does not notice - it keeps answering, just as confidently, and just as wrongly. **Data drift**
is the name for that movement, and this notebook is how we look for it.

The tool is [Evidently](https://docs.evidentlyai.com). It compares two datasets column by
column and reports where their distributions have separated.

## What we compare

Drift detection always needs **two** datasets: a **reference** (the world the model knows) and
a **current** one (the world as it is now). We build three, so we can see both a healthy result
and an unhealthy one:

| Dataset | What it is | What we expect |
| --- | --- | --- |
| **reference** | the training split | - |
| **current** | the test split | **no drift** - same shuffle, same world |
| **production** | a simulated future book of business | **drift** - deliberately shifted |

Only running the production comparison would be a trap. A report full of red tells you nothing
unless you have also seen the same report come back green on data you know is fine.

> **A word about the production data.** This project uses a static Kaggle file, so there is no
> real incoming traffic. Section 5.3 *invents* the production set by shifting the data on
> purpose. The drift we find there is drift we put there. That is honest for learning the tool,
> but it is not evidence about the real world.

## 5.1 Setup

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

TARGET = "charges"
NUMERIC_FEATURES = ["age", "bmi", "children"]
CATEGORICAL_FEATURES = ["sex", "smoker", "region"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print(f"Project root : {PROJECT_ROOT}")
print(f"Reports go to: {REPORT_DIR}")

Project root : C:\Users\ILLEGEAR\OneDrive\Desktop\Personal Project\DS & ML\Insurance Premium Prediction
Reports go to: C:\Users\ILLEGEAR\OneDrive\Desktop\Personal Project\DS & ML\Insurance Premium Prediction\reports


## 5.2 Reference and current

These come straight from the pipeline, so the split matches exactly what the model was trained
on. `Cleaner` and `Trainer` are the same classes `main.py` uses - importing them rather than
re-writing the split is what keeps this notebook honest.

`Predictor.predict()` returns dollars, having already undone the log transform. Evidently needs
the predictions as an ordinary column, so we attach them.

In [2]:
from steps.clean import Cleaner
from steps.ingest import Ingestion
from steps.predict import Predictor
from steps.train import Trainer

data = Cleaner(target=TARGET).clean_data(Ingestion().load_data())

trainer = Trainer()
X, y = trainer.feature_target_separator(data)
X_train, X_test, y_train, y_test = trainer.train_test_split_data(X, y)

predictor = Predictor()

reference = X_train.copy()
reference[TARGET] = y_train
reference["prediction"] = predictor.predict(X_train)

current = X_test.copy()
current[TARGET] = y_test
current["prediction"] = predictor.predict(X_test)

print(f"reference : {len(reference):>5} rows")
print(f"current   : {len(current):>5} rows")
print(f"model     : {predictor.model_name}")
reference.head()

reference :  1069 rows
current   :   268 rows
model     : RandomForestRegressor


,age,sex,bmi,children,smoker,region,charges,prediction
1113,23,male,24.510,0,no,northeast,2396.09590,2675.308065
967,21,male,25.745,2,no,northeast,3279.86855,4801.517176
598,52,female,37.525,2,no,northwest,33471.97189,14653.035069
170,63,male,41.470,0,no,southeast,13405.39030,14223.145973
275,47,female,26.600,2,no,northeast,9715.84100,10597.986541


## 5.3 Simulating a production dataset

Three changes, each one something that genuinely happens to an insurance book:

| Change | From | To | The story |
| --- | --- | --- | --- |
| `age` | mean 39.2 | mean ~45 | the book ages; fewer young people join |
| `bmi` | mean 30.7 | mean ~33.7 | population-wide weight gain |
| `smoker` | 20.5% yes | ~32% yes | a worse risk mix than we priced for |
| `region` | ~25% each | southeast ~45% | the company expanded into one region |

**`charges` is dropped on purpose.** In production you have the features the moment someone
applies, but you do not learn their real medical costs until the year is over - often much
later. That gap is the central difficulty of monitoring a live model, and pretending we have
the answers would teach the wrong lesson.

Everything is seeded, so re-running this cell reproduces the same file.

In [3]:
SEED = 42
N_PRODUCTION = 400

rng = np.random.default_rng(SEED)
base = pd.read_csv(DATA_DIR / "merged_data.csv")

production = base.sample(n=N_PRODUCTION, replace=True, random_state=SEED).reset_index(drop=True)

# 1. The book ages. Clipped to the range the model was trained on, so any drift
#    Evidently finds is a real distribution shift and not an out-of-range artefact.
production["age"] = (
    (production["age"] + rng.normal(6, 3, N_PRODUCTION)).round().clip(18, 64).astype(int)
)

# 2. Population-wide weight gain
production["bmi"] = (
    (production["bmi"] + rng.normal(3, 1.5, N_PRODUCTION)).clip(15.96, 53.13).round(2)
)

# 3. A worse smoking mix: flip non-smokers to smokers until the rate reaches 32%
n_to_flip = int(N_PRODUCTION * 0.32) - int((production["smoker"] == "yes").sum())
non_smokers = production.index[production["smoker"] == "no"]
production.loc[rng.choice(non_smokers, size=n_to_flip, replace=False), "smoker"] = "yes"

# 4. Expansion into the southeast
n_to_move = int(N_PRODUCTION * 0.45) - int((production["region"] == "southeast").sum())
elsewhere = production.index[production["region"] != "southeast"]
production.loc[rng.choice(elsewhere, size=n_to_move, replace=False), "region"] = "southeast"

# The true charges are not knowable yet - see the note above
production = production.drop(columns=[TARGET])

production.to_csv(DATA_DIR / "production.csv", index=False)

comparison = pd.DataFrame({
    "reference": [
        base["age"].mean(), base["bmi"].mean(),
        (base["smoker"] == "yes").mean() * 100, (base["region"] == "southeast").mean() * 100,
    ],
    "production": [
        production["age"].mean(), production["bmi"].mean(),
        (production["smoker"] == "yes").mean() * 100,
        (production["region"] == "southeast").mean() * 100,
    ],
}, index=["age (mean)", "bmi (mean)", "smoker yes (%)", "southeast (%)"]).round(1)

print(f"Wrote {DATA_DIR / 'production.csv'}  ({len(production)} rows, {TARGET} withheld)\n")
comparison

Wrote C:\Users\ILLEGEAR\OneDrive\Desktop\Personal Project\DS & ML\Insurance Premium Prediction\data\production.csv  (400 rows, charges withheld)



,reference,production
age (mean),39.2,44.4
bmi (mean),30.7,33.3
smoker yes (%),20.5,32.0
southeast (%),27.2,45.0


The model still scores this data happily. It has no way to know the population moved - that is
exactly the point.

In [4]:
production_scored = production.copy()
production_scored["prediction"] = predictor.predict_records(production.to_dict("records"))

print(f"reference  mean prediction : ${reference['prediction'].mean():>9,.0f}")
print(f"current    mean prediction : ${current['prediction'].mean():>9,.0f}")
print(f"production mean prediction : ${production_scored['prediction'].mean():>9,.0f}")
production_scored.head()

reference  mean prediction : $   12,449
current    mean prediction : $   13,792
production mean prediction : $   18,139


,age,sex,bmi,children,smoker,region,prediction
0,62,male,32.63,0,no,southwest,13561.917063
1,40,female,50.90,2,yes,southwest,43264.335722
2,64,male,29.41,0,no,southeast,13911.450866
3,48,female,26.28,5,no,southeast,10943.015422
4,18,female,35.13,4,yes,northeast,36650.427310


## 5.4 Describing the data to Evidently

`DataDefinition` is how Evidently learns which column is which. It replaced the old
`ColumnMapping`, which is what both reference projects use and what was removed in Evidently
0.7 (April 2025).

We need **two** definitions, because the two comparisons are not the same shape:

- **with a target** - reference and current both know the true `charges`, so error metrics
  like RMSE can be computed
- **without a target** - production does not, so only the inputs and the predictions can be
  compared

In [5]:
from evidently import DataDefinition, Dataset, Regression, Report
from evidently.presets import DataDriftPreset, DataSummaryPreset, RegressionPreset

definition_with_target = DataDefinition(
    numerical_columns=NUMERIC_FEATURES + [TARGET, "prediction"],
    categorical_columns=CATEGORICAL_FEATURES,
    regression=[Regression(target=TARGET, prediction="prediction")],
)

definition_no_target = DataDefinition(
    numerical_columns=NUMERIC_FEATURES + ["prediction"],
    categorical_columns=CATEGORICAL_FEATURES,
)

reference_ds = Dataset.from_pandas(reference, data_definition=definition_with_target)
current_ds = Dataset.from_pandas(current, data_definition=definition_with_target)

reference_features_ds = Dataset.from_pandas(
    reference[FEATURES + ["prediction"]], data_definition=definition_no_target
)
production_ds = Dataset.from_pandas(production_scored, data_definition=definition_no_target)

print("Four Evidently datasets built.")

Four Evidently datasets built.


## 5.5 Report A - the healthy baseline

Reference against current. Both come from the same shuffle of the same file, so there is no
reason for anything to have drifted. **This report should come back clean**, and if it does not,
the problem is our setup rather than the data.

`include_tests=True` adds a pass/fail tab beside the charts. Before Evidently 0.7 these were
two separate files.

Because both datasets carry the true `charges`, `RegressionPreset` can report real error -
the same RMSE and MAE `main.py` prints.

In [6]:
baseline_report = Report(
    [DataDriftPreset(), DataSummaryPreset(), RegressionPreset()],
    include_tests=True,
)
baseline_result = baseline_report.run(current_data=current_ds, reference_data=reference_ds)

baseline_path = REPORT_DIR / "baseline_drift.html"
baseline_result.save_html(str(baseline_path))

print(f"Saved {baseline_path.name}  ({baseline_path.stat().st_size / 1e6:.1f} MB)")
print("Open it in a browser to see the charts.")

Saved baseline_drift.html  (4.2 MB)
Open it in a browser to see the charts.


> **Why the report is not shown inline.** Rendering an Evidently result inside a notebook
> embeds the whole thing - about 5 MB - into the `.ipynb` file, which then lands in git on every
> run. That is the same problem `models/` and `data/` were moved to DVC to avoid. The HTML file
> is the artefact; open it in a browser.

## 5.6 Report B - the alarm

Reference against production. We know the answer in advance: `age`, `bmi`, `smoker` and
`region` were all moved on purpose in 5.3, so all four should light up.

Two things to look for beyond the obvious:

1. **`prediction` drifts too.** Nobody touched it. It moved because the inputs moved, and the
   model faithfully followed them. In a real system, where you cannot see the truth for months,
   prediction drift is often the earliest warning you get.
2. **No error metrics.** `RegressionPreset` is absent here - without the true `charges` there is
   nothing to be right or wrong about. Drift is a **proxy**: it tells you the world changed, not
   that the model got worse.

In [7]:
production_report = Report(
    [DataDriftPreset(), DataSummaryPreset()],
    include_tests=True,
)
production_result = production_report.run(
    current_data=production_ds, reference_data=reference_features_ds
)

production_path = REPORT_DIR / "production_drift.html"
production_result.save_html(str(production_path))

print(f"Saved {production_path.name}  ({production_path.stat().st_size / 1e6:.1f} MB)")
print("Open it in a browser to see the charts.")

Saved production_drift.html  (4.1 MB)
Open it in a browser to see the charts.


## 5.7 Reading the result in code

The HTML is for humans. The same numbers come back as a dict, which is what a scheduled job
would check - and, later, what CI could fail a build on.

In [8]:
def drift_summary(result, label):
    """Pull the per-column drift verdicts out of a finished Evidently run.

    Each `ValueDrift` metric carries a score and the threshold it was judged
    against. The method differs by column type - Evidently picks Wasserstein
    distance for numeric columns and Jensen-Shannon for categorical ones - so
    the scores are only comparable against their own threshold, never against
    each other.

    Args:
        result (evidently.Run): What `Report.run()` returned.
        label (str): Name for the comparison, used in the printed heading.

    Returns:
        pandas.DataFrame: One row per column, worst drift first, with the
        score, the threshold and the verdict.
    """
    report = result.dict()

    rows = []
    overall = None
    for metric in report["metrics"]:
        config = metric.get("config", {})
        kind = config.get("type", "")

        if kind.endswith("ValueDrift"):
            score = float(metric["value"])
            threshold = float(config["threshold"])
            rows.append({
                "column": config["column"],
                "method": config["method"],
                "score": round(score, 4),
                "threshold": threshold,
                "verdict": "DRIFT" if score > threshold else "ok",
            })
        elif kind.endswith("DriftedColumnsCount"):
            overall = metric["value"]

    frame = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

    failed = [t for t in report["tests"] if t["status"] == "FAIL"]
    print(f"\n=== {label} ===")
    if overall is not None:
        print(f"  {int(overall['count'])} of {len(rows)} columns drifted "
              f"(share {overall['share']:.1%}; dataset drift is declared at 50%)")
    print(f"  {len(failed)} of {len(report['tests'])} tests failed")

    return frame


display(drift_summary(baseline_result, "A: reference vs current (expect no drift)"))
display(drift_summary(production_result, "B: reference vs production (expect drift)"))


=== A: reference vs current (expect no drift) ===
  3 of 8 columns drifted (share 37.5%; dataset drift is declared at 50%)
  9 of 75 tests failed


,column,method,score,threshold,verdict
0,prediction,Wasserstein distance (normed),0.1287,0.1,DRIFT
1,bmi,Wasserstein distance (normed),0.1181,0.1,DRIFT
2,charges,Wasserstein distance (normed),0.1103,0.1,DRIFT
3,children,Wasserstein distance (normed),0.0494,0.1,ok
4,age,Wasserstein distance (normed),0.0392,0.1,ok
5,sex,Jensen-Shannon distance,0.0340,0.1,ok
6,region,Jensen-Shannon distance,0.0339,0.1,ok
7,smoker,Jensen-Shannon distance,0.0205,0.1,ok



=== B: reference vs production (expect drift) ===
  4 of 7 columns drifted (share 57.1%; dataset drift is declared at 50%)
  18 of 59 tests failed


,column,method,score,threshold,verdict
0,prediction,Wasserstein distance (normed),0.5153,0.1,DRIFT
1,bmi,Wasserstein distance (normed),0.4499,0.1,DRIFT
2,age,Wasserstein distance (normed),0.3721,0.1,DRIFT
3,region,Jensen-Shannon distance,0.1374,0.1,DRIFT
4,smoker,Jensen-Shannon distance,0.0969,0.1,ok
5,children,Wasserstein distance (normed),0.0517,0.1,ok
6,sex,Jensen-Shannon distance,0.0120,0.1,ok


### Two things in those tables worth stopping on

**1. The healthy baseline is not perfectly clean.** Report A flagged `prediction` (0.129),
`bmi` (0.118) and `charges` (0.110) as drifted - all barely over the 0.1 threshold. Nothing
changed between those two datasets; they are two halves of one shuffle. That is ordinary
sampling noise across 268 test rows, and it is why the **dataset-level share** is the number to
watch: 37.5% is below the 50% line, so Evidently correctly declares no dataset drift. Judging
single columns against a fixed threshold produces false alarms.

**2. Evidently nearly missed a change we made on purpose.** We moved `smoker` from 20.5% to
32% - a large, deliberate shift - and Report B scored it **0.0969, just under the threshold, and
called it `ok`**. Jensen-Shannon distance on a two-value column is blunt; it takes a lot of
movement to register. Meanwhile `bmi`, which we nudged by only 3 points, scored 0.45.

The lesson is not that the tool is broken. It is that a drift score is evidence, not a verdict,
and it is worth knowing which of your columns the default method is bad at. For a binary column
that matters this much to the price, watching the raw rate directly would catch what the
distance metric shrugs at.

## 5.8 What to do about it

Finding drift is not the same as needing to retrain. The report tells you the world moved; it
does not tell you the model got worse. Those are different claims, and only ground truth
settles the second one.

A reasonable order of response:

1. **Check it is real.** A shifted `region` mix might just be one big new client, not a change
   in the underlying population.
2. **Wait for ground truth where you can.** Once real `charges` arrive for some production rows,
   run Report A's setup on them. That gives actual RMSE, which is evidence rather than a proxy.
3. **Retrain on data that includes the new world.** `main.py` already does this - `dvc add data
   models`, commit, `dvc push`, and the new version is recorded.
4. **Re-run this notebook.** The new model becomes the reference, and drift should be gone.

### Where the reports live

`reports/` is gitignored. These HTML files are regenerated on every run, which is the same
reason `models/` and `data/` were moved out of git and into DVC. Re-running this notebook is
how you get them back.